# LangChain Customer Support Agent with Email

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shanjai-raj/commune-cookbook/blob/main/notebooks/langchain_customer_support.ipynb)

Build a complete customer support agent: it reads inbound emails, classifies the issue, generates a reply using an LLM, and sends it back in the same thread — all using LangChain and Commune.

**What you'll build:**

```
Customer email → Commune webhook → LangChain agent
                                        ↓
                              Classify issue type
                                        ↓
                              Generate empathetic reply
                                        ↓
                       Commune sends reply (same thread) → Customer sees reply
```

**Prerequisites:**
- [Commune API key](https://commune.email) (free tier)
- [OpenAI API key](https://platform.openai.com) (or substitute your preferred LLM)

In [ ]:
!pip install commune-mail langchain langchain-openai -q
print("\u2713 Dependencies installed")

## Setup

Set your API keys. In Google Colab, use **Secrets** (lock icon in the sidebar) to store `COMMUNE_API_KEY` and `OPENAI_API_KEY` securely.

In [ ]:
import os
from commune import CommuneClient
from langchain_openai import ChatOpenAI

COMMUNE_API_KEY = os.environ.get("COMMUNE_API_KEY", "comm_your_key_here")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-your_key_here")

commune = CommuneClient(api_key=COMMUNE_API_KEY)
llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0.3)

# Create a dedicated support inbox
inbox = commune.inboxes.create(local_part="support-agent")
print(f"\u2713 Support inbox ready: {inbox.address}")
print(f"  Inbox ID: {inbox.id}")

## Define Commune Tools for LangChain

We wrap the Commune SDK operations as `@tool`-decorated functions. LangChain's agent will decide which tools to call based on the current task.

The four tools below give the agent full read/write access to the inbox:
- `send_email` — send a new email or start a thread
- `reply_in_thread` — reply inside an existing conversation
- `read_inbox` — list recent threads
- `search_threads` — semantic search across thread history

In [ ]:
from langchain_core.tools import tool

INBOX_ID = inbox.id  # captured from setup above


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send a new email from the support inbox to a customer.

    Args:
        to: Recipient email address.
        subject: Email subject line.
        body: Plain-text email body.

    Returns:
        Confirmation string with message ID.
    """
    result = commune.messages.send(
        to=to,
        subject=subject,
        text=body,
        inbox_id=INBOX_ID,
    )
    return f"Email sent. Message ID: {result.get('message_id', 'ok')}"


@tool
def reply_in_thread(thread_id: str, to: str, body: str) -> str:
    """Reply to a customer within an existing email thread.

    Use this tool when responding to an inbound customer email so the reply
    appears in the same conversation thread in the customer's email client.

    Args:
        thread_id: The Commune thread ID (from the webhook payload or threads.list).
        to: Recipient email address (the customer's email).
        body: Plain-text reply body.

    Returns:
        Confirmation string.
    """
    result = commune.messages.send(
        to=to,
        text=body,
        inbox_id=INBOX_ID,
        thread_id=thread_id,
    )
    return f"Reply sent in thread {thread_id}. Message ID: {result.get('message_id', 'ok')}"


@tool
def read_inbox(limit: int = 10) -> str:
    """List the most recent email threads in the support inbox.

    Args:
        limit: Maximum number of threads to return (1-50).

    Returns:
        Formatted list of threads with subject, sender, and message count.
    """
    threads = commune.threads.list(inbox_id=INBOX_ID, limit=limit)
    if not threads.data:
        return "No threads found in inbox."

    lines = []
    for t in threads.data:
        lines.append(
            f"- Thread ID: {t.thread_id} | Subject: {t.subject or '(no subject)'} "
            f"| Messages: {t.message_count} | Last: {t.last_message_at}"
        )
    return "\n".join(lines)


@tool
def search_threads(query: str, limit: int = 5) -> str:
    """Search the support inbox using natural language semantic search.

    Use this to find past threads relevant to a customer's issue. Results are
    ranked by semantic similarity — not keyword matching.

    Args:
        query: Natural language search query, e.g. 'billing issue refund request'.
        limit: Maximum number of results to return.

    Returns:
        Formatted list of matching threads.
    """
    results = commune.search.threads(
        query=query,
        inbox_id=INBOX_ID,
        limit=limit,
    )
    if not results:
        return "No matching threads found."

    lines = []
    for r in results:
        lines.append(
            f"- Thread ID: {r.thread_id} | Subject: {r.subject} "
            f"| Similarity: {r.score:.2f}"
        )
    return "\n".join(lines)


tools = [send_email, reply_in_thread, read_inbox, search_threads]
print(f"\u2713 {len(tools)} tools registered: {[t.name for t in tools]}")

## Create the LangChain Support Agent

We use `create_openai_functions_agent` with a system prompt tuned for customer support. The agent can call any of the four tools above during its reasoning loop.

The `AgentExecutor` runs the agent until it reaches a final answer (or hits `max_iterations`).

In [ ]:
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYSTEM_PROMPT = """You are a helpful customer support agent for a SaaS company.

When you receive a customer email, you must:
1. Classify the issue type: billing, technical, account, general, or feature_request
2. Search for related past threads to understand if this customer has written before
3. Generate an empathetic, helpful reply that addresses the customer's concern
4. Reply in the same thread using reply_in_thread — never open a new thread for an existing conversation

Tone guidelines:
- Warm and empathetic — acknowledge the customer's frustration first
- Specific — reference their actual issue, not a generic template
- Actionable — always end with a clear next step
- Brief — aim for 3-4 sentences maximum

Always use the available tools rather than making up information.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_openai_functions_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=6,
    handle_parsing_errors=True,
)

print("\u2713 LangChain support agent ready")

## Webhook Handler (Flask)

In production, Commune POSTs to your webhook URL every time an email arrives. The handler below verifies the signature, extracts the email payload, and runs the agent.

Run this cell to see the full handler code — it won't start a server in this notebook environment.

In [ ]:
webhook_handler_code = '''
import json
import os
from flask import Flask, request, Response
from commune import verify_signature, WebhookVerificationError

app = Flask(__name__)

@app.route("/webhook", methods=["POST"])
def handle_inbound_email():
    body = request.get_data()  # raw bytes — required for HMAC verification

    # Step 1: Verify the webhook signature
    try:
        verify_signature(
            payload=body,
            signature=request.headers.get("x-commune-signature", ""),
            secret=os.environ["COMMUNE_WEBHOOK_SECRET"],
            timestamp=request.headers.get("x-commune-timestamp", ""),
        )
    except WebhookVerificationError as e:
        return Response(str(e), status=401)

    # Step 2: Parse the payload
    payload = json.loads(body)
    sender = payload["sender"]
    thread_id = payload["thread_id"]
    subject = payload.get("subject", "(no subject)")
    content = payload["content"]

    # Step 3: Run the LangChain agent
    task = (
        f"New support email received.\n"
        f"From: {sender}\n"
        f"Subject: {subject}\n"
        f"Thread ID: {thread_id}\n"
        f"Message: {content}\n\n"
        f"Please classify this issue, search for any prior context, "
        f"and reply to the customer using reply_in_thread."
    )
    agent_executor.invoke({"input": task})

    # Always return 200 quickly — Commune expects a fast response
    return {"ok": True}

if __name__ == "__main__":
    app.run(port=8000)
'''

print("Webhook handler:")
print(webhook_handler_code)

## Test the Agent Locally

We simulate an inbound email payload and run the agent directly — no webhook needed. Replace the `sender` and `content` fields to test different scenarios.

The agent will:
1. Search for prior threads from this sender
2. Classify the issue
3. Generate a reply
4. Call `reply_in_thread` (which will send the actual email if `thread_id` is valid)

In [ ]:
# Simulate an inbound email event (same structure as the webhook payload)
simulated_payload = {
    "sender": "alice@example.com",
    "thread_id": "thread_demo_001",
    "subject": "Can't log in — password reset not working",
    "content": (
        "Hi, I've been trying to reset my password for the past hour. "
        "The reset email never arrives, even though I've tried three times. "
        "I have a presentation in 2 hours and need access urgently. Please help!"
    ),
}

task_input = (
    f"New support email received.\n"
    f"From: {simulated_payload['sender']}\n"
    f"Subject: {simulated_payload['subject']}\n"
    f"Thread ID: {simulated_payload['thread_id']}\n"
    f"Message: {simulated_payload['content']}\n\n"
    f"Please classify this issue, search for any prior context from this customer, "
    f"and draft a reply. Since this is a simulation, show me the reply you would send "
    f"but do not actually call reply_in_thread."
)

print("Running agent...")
print("=" * 60)
result = agent_executor.invoke({"input": task_input})
print("=" * 60)
print("\nFinal answer:")
print(result["output"])

## What's next?

- **Deploy the webhook** — use [Railway](https://railway.app), [Render](https://render.com), or [ngrok](https://ngrok.com) for local testing
- **Register the webhook URL** in the [Commune dashboard](https://commune.email) under your inbox settings
- **Add memory** — use `ConversationBufferMemory` to track multi-turn conversations
- **Escalation** — add a `escalate_to_human` tool that sends an SMS alert via `commune.sms.send()`
- **[CrewAI notebook](./crewai_multi_agent_crew.ipynb)** — coordinate a triage + specialist + QA crew

**Commune docs:** https://commune.email/docs  
**LangChain docs:** https://python.langchain.com/docs